In [ ]:
# ========================
# IA Hierárquica: CONVERSAO -> FATOR
# ========================

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# ---------- 1) Ler base ----------
tabela = pd.read_excel("Banco_de_Fardos.xlsx")
tabela.columns = tabela.columns.str.strip()
tabela = tabela[["MATERIAL", "NOME_CONCO", "CONVERSAO", "FATOR"]].copy()

# ---------- 2) Preprocess ----------
tabela["FATOR"] = pd.to_numeric(tabela["FATOR"], errors="coerce").fillna(1.0)

# Encoders
enc_m = LabelEncoder(); tabela["MATERIAL_enc"] = enc_m.fit_transform(tabela["MATERIAL"].astype(str))
enc_n = LabelEncoder(); tabela["NOME_enc"]     = enc_n.fit_transform(tabela["NOME_CONCO"].astype(str))
enc_conv = LabelEncoder(); tabela["CONV_enc"] = enc_conv.fit_transform(tabela["CONVERSAO"].astype(str))

# Features e targets
X = tabela[["MATERIAL_enc", "NOME_enc"]]
y_conv = tabela["CONV_enc"]   # classificação
y_fator = tabela["FATOR"]     # regressão

# ---------- 3) Treinar ----------
modelo_conv = RandomForestClassifier(random_state=42, n_estimators=150)
modelo_conv.fit(X, y_conv)

# Modelo de fator condicionado à conversão
X_reg = X.copy()
X_reg["CONV_enc"] = y_conv
modelo_fator = RandomForestRegressor(random_state=42, n_estimators=150)
modelo_fator.fit(X_reg, y_fator)

print("Treinamento hierárquico concluído.")

# ---------- 4) Previsão ----------
novos = pd.read_excel("novos_fardos.xlsx")
novos.columns = novos.columns.str.strip()
novos = novos[["MATERIAL", "NOME_CONCO"]].copy()

def safe_map(series, encoder):
    m = {v:i for i,v in enumerate(encoder.classes_)}
    return series.astype(str).map(m).fillna(-1).astype(int)

novos["MATERIAL_enc"] = safe_map(novos["MATERIAL"], enc_m)
novos["NOME_enc"] = safe_map(novos["NOME_CONCO"], enc_n)

X_novos = novos[["MATERIAL_enc", "NOME_enc"]]

# Predição hierárquica
pred_conv_enc = modelo_conv.predict(X_novos)
X_novos_reg = X_novos.copy()
X_novos_reg["CONV_enc"] = pred_conv_enc
pred_fator = modelo_fator.predict(X_novos_reg)

# Decodificar conversão
pred_conv_labels = enc_conv.inverse_transform(
    np.clip(np.round(pred_conv_enc).astype(int), 0, len(enc_conv.classes_)-1)
)

# Resultado final
resultado = novos.copy()
resultado["CONVERSAO"] = pred_conv_labels
resultado["FATOR"] = pred_fator

resultado.to_excel("PlanilhaAtualizada_hierarquica.xlsx", index=False)
print("Resultado salvo: PlanilhaAtualizada_hierarquica.xlsx")
